# 13 Split Folds Make Patches Dataset 2

This notebook creates the fixed five-fold partition for dataset 2 and exports the real `224x224` patches and the cocci-only `256x256` GAN source patches.


## Why This Order Matters

We assign image-level folds first and only then cut patches. This keeps each original image fully inside one fold.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import json
import random
import shutil

import pandas as pd

from IPython.display import display
from PIL import Image

# This helper keeps the notebook easy to run from the repo root or from notebooks/.
def find_repo_root(start_path: Path) -> Path:
    if (start_path / "raw_data").exists():
        return start_path
    if start_path.name == "notebooks" and (start_path.parent / "raw_data").exists():
        return start_path.parent
    raise FileNotFoundError("Could not find the FYP2 repo root.")

REPO_ROOT = find_repo_root(Path.cwd())
MANIFESTS_DIR = REPO_ROOT / "manifests"
PATCHES_224_DIR = REPO_ROOT / "patches_224" / "dataset2"
GAN_PATCHES_256_DIR = REPO_ROOT / "gan_patches_256" / "dataset2"
RESULTS_DIR = REPO_ROOT / "results"
NOTEBOOK_TAG = "13_split_folds_make_patches_dataset2"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / NOTEBOOK_TAG
SEED = 2026
PATCH_SIZE_224 = 224
PATCH_SIZE_256 = 256

MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
PATCHES_224_DIR.mkdir(parents=True, exist_ok=True)
GAN_PATCHES_256_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# This helper reads a CSV file into a plain list of rows.
def read_csv_rows(csv_path: Path):
    with csv_path.open("r", newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

# This helper writes a CSV file with a stable header.
def write_csv_rows(csv_path: Path, rows, fieldnames):
    if not rows:
        raise ValueError(f"No rows were provided for {csv_path.name}.")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# This helper writes one small JSON file with clean formatting.
def write_json(json_path: Path, data):
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

# This helper stores paths in repo-relative form so the outputs stay portable.
def as_repo_relative(repo_root: Path, path_value: Path) -> str:
    return path_value.relative_to(repo_root).as_posix()

# This helper avoids divide-by-zero problems in small summary tables.
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

# This helper assigns rows to folds by binary label.
# It keeps the split focused on the minority-versus-majority framing of the study.
def assign_rows_to_folds(rows, label_key, sample_key, seed_value, fold_count=5):
    rows_by_label = defaultdict(list)
    for row in rows:
        rows_by_label[row[label_key]].append(dict(row))

    assigned_rows = []

    for label_index, label_value in enumerate(sorted(rows_by_label)):
        label_rows = sorted(rows_by_label[label_value], key=lambda row: row[sample_key])
        random.Random(seed_value + label_index).shuffle(label_rows)
        next_fold_id = 1

        for row in label_rows:
            row_copy = dict(row)
            row_copy["fold_id"] = next_fold_id
            assigned_rows.append(row_copy)
            next_fold_id = (next_fold_id % fold_count) + 1

    if len(assigned_rows) != len(rows):
        raise AssertionError("The fold assignment lost some rows.")

    return assigned_rows

In [3]:
# We load the clean dataset 2 image manifest and assign one raw fold id to every image.
# The split only follows the binary labels because this stage is framed as an imbalance study.
image_rows = read_csv_rows(MANIFESTS_DIR / "dataset2_image_manifest.csv")
for row in image_rows:
    row["binary_label"] = int(row["binary_label"])

fold_rows = assign_rows_to_folds(
    rows=image_rows,
    label_key="binary_label",
    sample_key="image_id",
    seed_value=SEED,
)

fold_manifest_rows = []
for row in sorted(fold_rows, key=lambda item: (item["fold_id"], item["image_id"])):
    fold_manifest_rows.append({
        "image_id": row["image_id"],
        "species_name_raw": row["species_name_raw"],
        "species_name": row["species_name"],
        "binary_label": row["binary_label"],
        "binary_group": row["binary_group"],
        "width": row["width"],
        "height": row["height"],
        "file_path": row["file_path"],
        "fold_id": row["fold_id"],
    })

fold_manifest_path = MANIFESTS_DIR / "dataset2_fold_manifest.csv"
write_csv_rows(fold_manifest_path, fold_manifest_rows, list(fold_manifest_rows[0].keys()))

fold_summary_rows = []
for fold_id in range(1, 6):
    rows = [row for row in fold_manifest_rows if int(row["fold_id"]) == fold_id]
    cocci_count = sum(1 for row in rows if int(row["binary_label"]) == 0)
    bacilli_count = sum(1 for row in rows if int(row["binary_label"]) == 1)
    fold_summary_rows.append({
        "fold_id": fold_id,
        "image_count": len(rows),
        "cocci_image_count": cocci_count,
        "bacilli_image_count": bacilli_count,
        "cocci_ratio": round(safe_divide(cocci_count, len(rows)), 6),
        "bacilli_ratio": round(safe_divide(bacilli_count, len(rows)), 6),
    })

species_fold_rows = []
for fold_id in range(1, 6):
    rows = [row for row in fold_manifest_rows if int(row["fold_id"]) == fold_id]
    counts = Counter(row["species_name"] for row in rows)
    for species_name in sorted(counts):
        species_fold_rows.append({
            "fold_id": fold_id,
            "species_name": species_name,
            "image_count": counts[species_name],
        })

actual_fold_sizes = sorted(row["image_count"] for row in fold_summary_rows)
if actual_fold_sizes != [85, 85, 85, 86, 86]:
    raise AssertionError(f"Expected dataset 2 fold sizes [85, 85, 85, 86, 86], but found {actual_fold_sizes}.")

write_csv_rows(NOTEBOOK_RESULTS_DIR / "fold_summary.csv", fold_summary_rows, list(fold_summary_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "species_by_fold.csv", species_fold_rows, list(species_fold_rows[0].keys()))
display(pd.DataFrame(fold_summary_rows))
display(pd.DataFrame(species_fold_rows))
print(f"Saved dataset 2 fold manifest to: {fold_manifest_path}")

,fold_id,image_count,cocci_image_count,bacilli_image_count,cocci_ratio,bacilli_ratio
0,1,86,36,50,0.418605,0.581395
1,2,86,36,50,0.418605,0.581395
2,3,85,36,49,0.423529,0.576471
3,4,85,36,49,0.423529,0.576471
4,5,85,36,49,0.423529,0.576471


,fold_id,species_name,image_count
0,1,Clostridium.perfringens,13
1,1,Enterococcus.faecalis,11
2,1,Enterococcus.faecium,12
3,1,Escherichia.coli,13
4,1,Listeria.monocytogenes,14
5,1,Pseudomonas.aeruginosa,10
6,1,Staphylococcus.aureus,13
7,2,Clostridium.perfringens,12
8,2,Enterococcus.faecalis,13
9,2,Enterococcus.faecium,8


Saved dataset 2 fold manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\dataset2_fold_manifest.csv


In [4]:
# We clear the dataset 2 patch folders before writing the new protocol outputs.
if PATCHES_224_DIR.exists():
    shutil.rmtree(PATCHES_224_DIR)
if GAN_PATCHES_256_DIR.exists():
    shutil.rmtree(GAN_PATCHES_256_DIR)
PATCHES_224_DIR.mkdir(parents=True, exist_ok=True)
GAN_PATCHES_256_DIR.mkdir(parents=True, exist_ok=True)

patch_rows_224 = []
gan_patch_rows_256 = []
patch_audit_rows = []
fold_patch_counts_224 = Counter()
fold_patch_counts_256 = Counter()

for fold_row in fold_manifest_rows:
    fold_id = int(fold_row["fold_id"])
    image_path = REPO_ROOT / fold_row["file_path"]
    with Image.open(image_path) as image_handle:
        rgba_image = image_handle.convert("RGBA")
        width, height = rgba_image.size

        candidate_count_224 = 0
        kept_count_224 = 0
        x_steps_224 = width // PATCH_SIZE_224
        y_steps_224 = height // PATCH_SIZE_224
        for y_index in range(y_steps_224):
            for x_index in range(x_steps_224):
                candidate_count_224 += 1
                x_start = x_index * PATCH_SIZE_224
                y_start = y_index * PATCH_SIZE_224
                patch_rgba = rgba_image.crop((x_start, y_start, x_start + PATCH_SIZE_224, y_start + PATCH_SIZE_224))
                if patch_rgba.getchannel("A").getextrema() != (255, 255):
                    continue
                patch_id = f"{fold_row['image_id'].replace('.', '_')}_{PATCH_SIZE_224}_{x_start}_{y_start}"
                target_dir = PATCHES_224_DIR / f"fold_{fold_id}"
                target_dir.mkdir(parents=True, exist_ok=True)
                patch_path = target_dir / f"{patch_id}.png"
                patch_rgba.convert("RGB").save(patch_path)
                patch_rows_224.append({
                    "dataset_name": "dataset2",
                    "patch_id": patch_id,
                    "image_id": fold_row["image_id"],
                    "species_name": fold_row["species_name"],
                    "binary_label": fold_row["binary_label"],
                    "binary_group": fold_row["binary_group"],
                    "fold_id": fold_id,
                    "patch_size": PATCH_SIZE_224,
                    "x_start": x_start,
                    "y_start": y_start,
                    "raw_path": fold_row["file_path"],
                    "file_path": as_repo_relative(REPO_ROOT, patch_path),
                    "source_type": "real",
                })
                kept_count_224 += 1
                fold_patch_counts_224[fold_id] += 1

        candidate_count_256 = 0
        kept_count_256 = 0
        if int(fold_row["binary_label"]) == 0:
            x_steps_256 = width // PATCH_SIZE_256
            y_steps_256 = height // PATCH_SIZE_256
            for y_index in range(y_steps_256):
                for x_index in range(x_steps_256):
                    candidate_count_256 += 1
                    x_start = x_index * PATCH_SIZE_256
                    y_start = y_index * PATCH_SIZE_256
                    patch_rgba = rgba_image.crop((x_start, y_start, x_start + PATCH_SIZE_256, y_start + PATCH_SIZE_256))
                    if patch_rgba.getchannel("A").getextrema() != (255, 255):
                        continue
                    patch_id = f"{fold_row['image_id'].replace('.', '_')}_{PATCH_SIZE_256}_{x_start}_{y_start}"
                    target_dir = GAN_PATCHES_256_DIR / f"fold_{fold_id}"
                    target_dir.mkdir(parents=True, exist_ok=True)
                    patch_path = target_dir / f"{patch_id}.png"
                    patch_rgba.convert("RGB").save(patch_path)
                    gan_patch_rows_256.append({
                        "dataset_name": "dataset2",
                        "gan_patch_id": patch_id,
                        "image_id": fold_row["image_id"],
                        "species_name": fold_row["species_name"],
                        "binary_label": fold_row["binary_label"],
                        "binary_group": fold_row["binary_group"],
                        "fold_id": fold_id,
                        "patch_size": PATCH_SIZE_256,
                        "x_start": x_start,
                        "y_start": y_start,
                        "raw_path": fold_row["file_path"],
                        "file_path": as_repo_relative(REPO_ROOT, patch_path),
                        "source_type": "real_cocci_for_gan",
                    })
                    kept_count_256 += 1
                    fold_patch_counts_256[fold_id] += 1

    patch_audit_rows.append({
        "image_id": fold_row["image_id"],
        "fold_id": fold_id,
        "species_name": fold_row["species_name"],
        "candidate_patch_count_224": candidate_count_224,
        "kept_patch_count_224": kept_count_224,
        "candidate_patch_count_256": candidate_count_256,
        "kept_patch_count_256": kept_count_256,
    })

patch_manifest_path = MANIFESTS_DIR / "dataset2_patch_manifest_224.csv"
gan_patch_manifest_path = MANIFESTS_DIR / "dataset2_gan_patch_manifest_256.csv"
write_csv_rows(patch_manifest_path, patch_rows_224, list(patch_rows_224[0].keys()))
write_csv_rows(gan_patch_manifest_path, gan_patch_rows_256, list(gan_patch_rows_256[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "patch_audit.csv", patch_audit_rows, list(patch_audit_rows[0].keys()))

patch_summary_rows = []
for fold_id in range(1, 6):
    patch_summary_rows.append({
        "fold_id": fold_id,
        "real_patch_count_224": fold_patch_counts_224[fold_id],
        "gan_source_patch_count_256": fold_patch_counts_256[fold_id],
    })

write_csv_rows(NOTEBOOK_RESULTS_DIR / "patch_summary.csv", patch_summary_rows, list(patch_summary_rows[0].keys()))
write_json(
    NOTEBOOK_RESULTS_DIR / "summary.json",
    {
        "notebook_tag": NOTEBOOK_TAG,
        "seed": SEED,
        "patch_size_224": PATCH_SIZE_224,
        "patch_size_256": PATCH_SIZE_256,
        "fold_manifest_path": as_repo_relative(REPO_ROOT, fold_manifest_path),
        "patch_manifest_path": as_repo_relative(REPO_ROOT, patch_manifest_path),
        "gan_patch_manifest_path": as_repo_relative(REPO_ROOT, gan_patch_manifest_path),
    },
)

display(pd.DataFrame(patch_summary_rows))
print(f"Saved dataset 2 patch manifest to: {patch_manifest_path}")
print(f"Saved dataset 2 GAN patch manifest to: {gan_patch_manifest_path}")

,fold_id,real_patch_count_224,gan_source_patch_count_256
0,1,870,260
1,2,837,239
2,3,806,229
3,4,821,243
4,5,845,240


Saved dataset 2 patch manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\dataset2_patch_manifest_224.csv
Saved dataset 2 GAN patch manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\dataset2_gan_patch_manifest_256.csv
